In [4]:
import pandas as pd
from pathlib import Path

# Raw price dataset
price_path = Path("../dataset/raw/price_dataset.csv")

price_df = pd.read_csv(price_path)

print("Dataset loaded successfully!")
print("Shape:", price_df.shape)

display(price_df.head(10))

Dataset loaded successfully!
Shape: (28, 6)


,category,price,unit,condition,source,retrieved_on
0,E-Waste,32.28,per_kg,not_specified,Scrapprice,9/23/2026
1,Li-ion Battery,313.75,per_kg,not_specified,Scrapprice,9/23/2026
2,PCB,394.08,per_kg,not_specified,Scrapprice,9/23/2026
3,Washing Machine,667.54,per_piece,not_specified,Scrapprice,9/23/2026
4,Mobile,300.00,per_piece,not_specified,Scrappbazaar,9/11/2026
5,Battery,85.00,per_kg,not_specified,Scrappbazaar,9/11/2026
6,Washing Machine,700.00,per_piece,varies_by_type_condition,Scrappbazaar,9/11/2026
7,Washing Machine,800.00,per_piece,fully_or_semi_automatic,Scrappbazaar,9/11/2026
8,TV,200.00,per_piece,varies_by_size_type,Scrappbazaar,9/11/2026
9,Microwave,300.00,per_piece,not_specified,Scrappbazaar,9/11/2026


In [5]:
print("Columns:")
print(price_df.columns.tolist())

print("\nData types:")
print(price_df.dtypes)

print("\nMissing values:")
print(price_df.isnull().sum())

Columns:
['category', 'price', 'unit', 'condition', 'source', 'retrieved_on']

Data types:
category            str
price           float64
unit                str
condition           str
source              str
retrieved_on        str
dtype: object

Missing values:
category        0
price           0
unit            0
condition       0
source          0
retrieved_on    0
dtype: int64


In [6]:
print("Categories:")
print(price_df["category"].value_counts())

print("\nUnits:")
print(price_df["unit"].value_counts())

print("\nSources:")
print(price_df["source"].value_counts())

Categories:
category
Washing Machine    8
Mobile             4
TV                 4
Microwave          4
Battery            3
Printer            2
E-Waste            1
Li-ion Battery     1
PCB                1
Name: count, dtype: int64

Units:
unit
per_piece    19
per_kg        9
Name: count, dtype: int64

Sources:
source
RecycleOne      12
Scrappbazaar     7
ScrapExpress     5
Scrapprice       4
Name: count, dtype: int64


In [7]:
print("PRICE SUMMARY")
print(price_df.groupby(["category", "unit"])["price"].agg(
    ["count", "min", "max", "mean"]
).round(2))

PRICE SUMMARY
                           count     min     max    mean
category        unit                                    
Battery         per_kg         3   70.00   95.00   83.33
E-Waste         per_kg         1   32.28   32.28   32.28
Li-ion Battery  per_kg         1  313.75  313.75  313.75
Microwave       per_piece      4  100.00  300.00  200.00
Mobile          per_piece      4   15.00  300.00  126.25
PCB             per_kg         1  394.08  394.08  394.08
Printer         per_kg         2   15.00   20.00   17.50
TV              per_kg         1   20.00   20.00   20.00
                per_piece      3   80.00  200.00  143.33
Washing Machine per_piece      8  350.00  800.00  614.69


In [8]:
print("\nUNIT-WISE DATA:")
display(
    price_df[["category", "price", "unit", "condition", "source"]]
    .sort_values(["category", "unit"])
)


UNIT-WISE DATA:


,category,price,unit,condition,source
5,Battery,85.00,per_kg,not_specified,Scrappbazaar
15,Battery,95.00,per_kg,not_specified,ScrapExpress
25,Battery,70.00,per_kg,without_water,RecycleOne
0,E-Waste,32.28,per_kg,not_specified,Scrapprice
1,Li-ion Battery,313.75,per_kg,not_specified,Scrapprice
9,Microwave,300.00,per_piece,not_specified,Scrappbazaar
13,Microwave,200.00,per_piece,not_specified,ScrapExpress
16,Microwave,100.00,per_piece,range_low,RecycleOne
17,Microwave,200.00,per_piece,range_high,RecycleOne
4,Mobile,300.00,per_piece,not_specified,Scrappbazaar


In [9]:
import pandas as pd
from pathlib import Path

# -----------------------------
# 1. Load raw price dataset
# -----------------------------
raw_path = Path("../dataset/raw/price_dataset.csv")
price_df = pd.read_csv(raw_path)

# -----------------------------
# 2. Categories supported by
#    our E-Waste image classifier
# -----------------------------
image_classes = [
    "Battery",
    "Keyboard",
    "Microwave",
    "Mobile",
    "Mouse",
    "PCB",
    "Player",
    "Printer",
    "Television",
    "Washing Machine"
]

# TV in price dataset -> Television
price_df["category"] = price_df["category"].replace({
    "TV": "Television"
})

# -----------------------------
# 3. Keep only categories that
#    our image model recognizes
# -----------------------------
processed_df = price_df[
    price_df["category"].isin(image_classes)
].copy()

# -----------------------------
# 4. Clean text columns
# -----------------------------
processed_df["category"] = processed_df["category"].str.strip()
processed_df["unit"] = processed_df["unit"].str.strip()
processed_df["condition"] = processed_df["condition"].str.strip()
processed_df["source"] = processed_df["source"].str.strip()

# -----------------------------
# 5. Create condition group
# -----------------------------
def condition_group(condition):

    condition = condition.lower()

    if "low" in condition:
        return "low"

    if "high" in condition:
        return "high"

    if "without_water" in condition:
        return "without_water"

    if "fully_or_semi" in condition:
        return "automatic"

    if "varies" in condition:
        return "varies"

    if "lcd" in condition:
        return "lcd"

    if "crt" in condition:
        return "crt"

    return "not_specified"


processed_df["condition_group"] = (
    processed_df["condition"]
    .apply(condition_group)
)

# -----------------------------
# 6. Make price numeric
# -----------------------------
processed_df["price"] = pd.to_numeric(
    processed_df["price"],
    errors="coerce"
)

# Remove invalid price rows
processed_df = processed_df.dropna(
    subset=["price"]
)

# -----------------------------
# 7. Reset index
# -----------------------------
processed_df = processed_df.reset_index(drop=True)

# -----------------------------
# 8. Save processed dataset
# -----------------------------
processed_path = Path(
    "../dataset/processed/price_dataset_processed.csv"
)

processed_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

processed_df.to_csv(
    processed_path,
    index=False
)

print("Processed dataset created successfully!")
print("Location:", processed_path)
print("Rows:", len(processed_df))

display(processed_df)

Processed dataset created successfully!
Location: ..\dataset\processed\price_dataset_processed.csv
Rows: 26


,category,price,unit,condition,source,retrieved_on,condition_group
0,PCB,394.08,per_kg,not_specified,Scrapprice,9/23/2026,not_specified
1,Washing Machine,667.54,per_piece,not_specified,Scrapprice,9/23/2026,not_specified
2,Mobile,300.00,per_piece,not_specified,Scrappbazaar,9/11/2026,not_specified
3,Battery,85.00,per_kg,not_specified,Scrappbazaar,9/11/2026,not_specified
4,Washing Machine,700.00,per_piece,varies_by_type_condition,Scrappbazaar,9/11/2026,varies
5,Washing Machine,800.00,per_piece,fully_or_semi_automatic,Scrappbazaar,9/11/2026,automatic
6,Television,200.00,per_piece,varies_by_size_type,Scrappbazaar,9/11/2026,varies
7,Microwave,300.00,per_piece,not_specified,Scrappbazaar,9/11/2026,not_specified
8,Printer,20.00,per_kg,not_specified,Scrappbazaar,9/11/2026,not_specified
9,Mobile,80.00,per_piece,not_specified,ScrapExpress,9/24/2026,not_specified


In [12]:
print("Total rows:", len(processed_df))

print("\nRows per category:")
print(processed_df["category"].value_counts())

print("\nRows per unit:")
print(processed_df["unit"].value_counts())

Total rows: 26

Rows per category:
category
Washing Machine    8
Mobile             4
Television         4
Microwave          4
Battery            3
Printer            2
PCB                1
Name: count, dtype: int64

Rows per unit:
unit
per_piece    19
per_kg        7
Name: count, dtype: int64


In [13]:
display(
    processed_df.sort_values(
        ["category", "unit", "price"]
    )[[
        "category",
        "price",
        "unit",
        "condition_group",
        "source"
    ]]
)

,category,price,unit,condition_group,source
23,Battery,70.00,per_kg,without_water,RecycleOne
3,Battery,85.00,per_kg,not_specified,Scrappbazaar
13,Battery,95.00,per_kg,not_specified,ScrapExpress
14,Microwave,100.00,per_piece,low,RecycleOne
11,Microwave,200.00,per_piece,not_specified,ScrapExpress
15,Microwave,200.00,per_piece,high,RecycleOne
7,Microwave,300.00,per_piece,not_specified,Scrappbazaar
24,Mobile,15.00,per_piece,low,RecycleOne
9,Mobile,80.00,per_piece,not_specified,ScrapExpress
25,Mobile,110.00,per_piece,high,RecycleOne


In [14]:
# Separate price data by unit

price_per_kg = processed_df[
    processed_df["unit"] == "per_kg"
].copy()

price_per_piece = processed_df[
    processed_df["unit"] == "per_piece"
].copy()

print("₹/kg records:", len(price_per_kg))
print("₹/piece records:", len(price_per_piece))

print("\n₹/kg categories:")
print(price_per_kg["category"].value_counts())

print("\n₹/piece categories:")
print(price_per_piece["category"].value_counts())

₹/kg records: 7
₹/piece records: 19

₹/kg categories:
category
Battery       3
Printer       2
PCB           1
Television    1
Name: count, dtype: int64

₹/piece categories:
category
Washing Machine    8
Mobile             4
Microwave          4
Television         3
Name: count, dtype: int64


In [15]:
print("₹/kg price range:")
print(price_per_kg["price"].describe())

print("\n₹/piece price range:")
print(price_per_piece["price"].describe())

₹/kg price range:
count      7.000000
mean      99.868571
std      133.951437
min       15.000000
25%       20.000000
50%       70.000000
75%       90.000000
max      394.080000
Name: price, dtype: float64

₹/piece price range:
count     19.000000
mean     350.133684
std      261.029619
min       15.000000
25%      130.000000
50%      300.000000
75%      550.000000
max      800.000000
Name: price, dtype: float64


In [17]:
# ============================================
# PRICE BASELINE CALCULATION
# ============================================

# Create separate datasets
price_kg = processed_df[
    processed_df["unit"] == "per_kg"
].copy()

price_piece = processed_df[
    processed_df["unit"] == "per_piece"
].copy()

# Calculate median price for each category
kg_baseline = (
    price_kg
    .groupby("category")["price"]
    .median()
    .sort_values()
)

piece_baseline = (
    price_piece
    .groupby("category")["price"]
    .median()
    .sort_values()
)

# Display results
print("===== ₹/KG BASELINE =====")
print(kg_baseline)

print("\n===== ₹/PIECE BASELINE =====")
print(piece_baseline)

===== ₹/KG BASELINE =====
category
Printer        17.50
Television     20.00
Battery        85.00
PCB           394.08
Name: price, dtype: float64

===== ₹/PIECE BASELINE =====
category
Mobile              95.00
Television         150.00
Microwave          200.00
Washing Machine    633.77
Name: price, dtype: float64
